# Colab 12 — ¿Se conserva realmente el momento en un choque?

**Laboratorio 1 · Departamento de Física · FCEN-UBA**

Clase 12 — 28/10

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/charlyacha/Labo1-colabs/blob/main/12_Choques_y_conservacion.ipynb)

El 'realmente' es el punto. La respuesta correcta no es 'sí': es '$\Delta p$ es compatible con cero dentro de $1{,}2\sigma$'. Pasar de *ley que se cumple* a *magnitud que se compara con su propia incerteza* es la madurez experimental que este curso quiere dejar.

**Al terminar vas a poder:** verificar cuantitativamente una ley de conservación y determinar un coeficiente de restitución por ajuste.

---

### Antes de tocar nada

Andá a **Archivo → Guardar una copia en Drive**. Vas a trabajar sobre tu copia:
lo que escribas acá sin copiar primero no se guarda en ningún lado.

Este cuaderno se recorre **de arriba hacia abajo**. Las celdas no son
independientes: cada una usa lo que definieron las anteriores. Si algo tira
`NameError`, casi siempre es porque salteaste una celda.

In [ ]:
import os

if not os.path.exists("lab1_utils.py"):
    !wget -q -O lab1_utils.py https://raw.githubusercontent.com/charlyacha/Labo1-colabs/main/lab1_utils.py

import numpy as np
import matplotlib.pyplot as plt
import lab1_utils as lab

lab.estilo_lab1()
print("Listo. numpy", np.__version__)

### 1. Los datos del choque

Dos carritos en riel de aire con photogates. De cada uno se conocen la masa
y la velocidad antes y después.

In [ ]:
generador = np.random.default_rng(12)

m1, sm1 = 0.2512, 0.0002
m2, sm2 = 0.4987, 0.0002

v1i, sv1i = 0.4123, 0.0031
v2i, sv2i = -0.1050, 0.0028
v1f, sv1f = -0.1902, 0.0030
v2f, sv2f = 0.1954, 0.0029

print("masas y velocidades cargadas")

### 2. Verificar la conservación, con incerteza

$$\Delta p = (m_1 v_{1f} + m_2 v_{2f}) - (m_1 v_{1i} + m_2 v_{2i})$$

Y su incerteza sale de propagar, con la observación de que cada término
aporta $(v\,\sigma_m)^2 + (m\,\sigma_v)^2$.

In [ ]:
def momento(m, sm, v, sv):
    p = m*v
    sp = np.sqrt((v*sm)**2 + (m*sv)**2)
    return p, sp


p1i, sp1i = momento(m1, sm1, v1i, sv1i)
p2i, sp2i = momento(m2, sm2, v2i, sv2i)
p1f, sp1f = momento(m1, sm1, v1f, sv1f)
p2f, sp2f = momento(m2, sm2, v2f, sv2f)

p_inicial = p1i + p2i
sp_inicial = np.sqrt(sp1i**2 + sp2i**2)
p_final = p1f + p2f
sp_final = np.sqrt(sp1f**2 + sp2f**2)

lab.reportar(p_inicial, sp_inicial, "kg·m/s", nombre="p inicial")
lab.reportar(p_final, sp_final, "kg·m/s", nombre="p final ")
print()

delta_p = p_final - p_inicial
s_delta = np.sqrt(sp_inicial**2 + sp_final**2)
lab.reportar(delta_p, s_delta, "kg·m/s", nombre="Δp")
print(f"Δp está a {abs(delta_p)/s_delta:.2f} sigma de cero.")

Ésa es la forma correcta de enunciar el resultado. No "se conserva" ni "da
parecido": **$\Delta p$ es compatible con cero dentro de $z$ sigma**.

Y notá el corolario incómodo, que conviene decir en voz alta: si tus barras
de error fueran diez veces más grandes, la conservación se "verificaría"
mucho mejor. Una verificación experimental vale lo que valen sus incertezas.
La afirmación fuerte no es "$\Delta p \approx 0$" sino "$\Delta p = 0$ dentro
de una precisión que hay que declarar", y que la celda que sigue calcula.

In [ ]:
print(f"Verificación al {100*s_delta/abs(p_inicial):.1f} % del momento inicial.")

### 3. La energía

En un choque el momento se conserva siempre; la energía cinética **no**,
salvo que el choque sea elástico. Cuánto se pierde caracteriza al choque.

In [ ]:
def energia(m, sm, v, sv):
    E = 0.5*m*v**2
    sE = np.sqrt((0.5*v**2*sm)**2 + (m*v*sv)**2)
    return E, sE


Ei = energia(m1, sm1, v1i, sv1i)[0] + energia(m2, sm2, v2i, sv2i)[0]
sEi = np.sqrt(energia(m1, sm1, v1i, sv1i)[1]**2
              + energia(m2, sm2, v2i, sv2i)[1]**2)
Ef = energia(m1, sm1, v1f, sv1f)[0] + energia(m2, sm2, v2f, sv2f)[0]
sEf = np.sqrt(energia(m1, sm1, v1f, sv1f)[1]**2
              + energia(m2, sm2, v2f, sv2f)[1]**2)

lab.reportar(Ei*1000, sEi*1000, "mJ", nombre="E inicial")
lab.reportar(Ef*1000, sEf*1000, "mJ", nombre="E final  ")

fraccion = Ef/Ei
s_fraccion = fraccion*np.sqrt((sEi/Ei)**2 + (sEf/Ef)**2)
print()
lab.reportar(100*fraccion, 100*s_fraccion, "%", nombre="E final / E inicial")

v_rel_i = v1i - v2i
v_rel_f = v1f - v2f
e_rest = abs(v_rel_f/v_rel_i)
print(f"\ncoeficiente de restitución e = |v_rel_f / v_rel_i| = {e_rest:.3f}")

La energía que falta no desapareció: se fue en deformación, en sonido y en
calor. Si el choque fuera perfectamente elástico, $e = 1$; si fuera
perfectamente plástico, $e = 0$ y los carritos saldrían pegados.

### 4. Restitución por otro camino: la pelota que rebota

Una pelota soltada desde $h_0$ rebota hasta $h_1 = e^2 h_0$, después hasta
$h_2 = e^4 h_0$, y en general

$$h_n = h_0\,(e^2)^n$$

Filmada con Tracker, las alturas máximas salen de los picos de la
trayectoria. Es un ajuste exponencial: **la misma herramienta de la Clase
10 en un contexto completamente distinto**, que es la mejor manera de
consolidar algo.

In [ ]:
n = np.arange(0, 8)
e_real = 0.82
h = 1.20*(e_real**2)**n * generador.normal(1, 0.02, size=len(n))
sh = 0.02*h


def decaimiento(n, h0, e2):
    return h0*e2**n


p, err, _ = lab.ajustar(decaimiento, n, h, yerr=sh, p0=[1.2, 0.7],
                        nombres=["h0 (m)", "e²"])

e_coef = np.sqrt(p[1])
se_coef = err[1]/(2*np.sqrt(p[1]))
print()
lab.reportar(e_coef, se_coef, nombre="coeficiente de restitución")

In [ ]:
fig, axes = lab.grafico_con_residuos(
    n, h, decaimiento, p, yerr=sh, normalizar_residuos=True,
    xlabel="Número de rebote, n", ylabel="Altura máxima (m)")
plt.show()

### 5. Una pregunta que vale la pena

¿Cuántas veces rebota la pelota antes de quedarse quieta? El tiempo entre
rebotes también decae geométricamente, y la suma de una serie geométrica
converge: **infinitos rebotes en tiempo finito**.

In [ ]:
h0, g = p[0], 9.797
t_total = np.sqrt(2*h0/g)*(1 + 2*e_coef/(1 - e_coef))

print(f"tiempo total hasta el reposo: {t_total:.2f} s")
print("con infinitos rebotes. Escuchá una pelota de ping pong caer:")
print("ese zumbido final es la serie convergiendo.")

### 6. Ejercicios

1. Verificá la conservación del momento con tus datos, en un choque elástico
   y en uno inelástico. Informá el resultado con la forma correcta:
   "$\Delta p$ compatible con cero al $x$ %".
2. ¿Qué medición domina la incerteza de $\Delta p$? Armá la tabla de
   contribuciones como en la Clase 4.
3. Filmá **la misma** pelota rebotando y determiná $e$ por los dos caminos:
   de las velocidades justo antes y después de un rebote, y del ajuste de la
   sucesión de alturas. Compará con un test de compatibilidad. (Ojo: los dos
   ejemplos de este cuaderno son sistemas distintos —carritos y pelota— así
   que no tiene sentido compararlos entre sí.)
4. **De diseño:** ¿cómo mejorarías el experimento para verificar la
   conservación con una precisión cinco veces mejor? Estimá antes de proponer: ¿es
   alcanzable con el equipamiento de la mesada?

In [ ]:
# Espacio de trabajo para los ejercicios.

### Entrega corta 3 — última antes del parcial

Verificación cuantitativa de la conservación del momento con incertezas.
Se entrega también la propuesta escrita de Práctica Especial.